# VISCERA · Expanded disease, shape and colour generation

Choose **Runtime → Change runtime type → A100 GPU** if available, then **Run all**.
Approve the Google Drive prompt when Google asks. No code edits or API keys are needed.
Other CUDA GPUs are supported; allocation and compute limits are controlled by Google.
This notebook does not buy compute or change your account plan.

Keep the three training ZIPs beside this notebook in **VISCERA Expanded Disease Training RC2 2026-10-07**.
The final cell downloads **viscera-joint-v2-weights.zip**, and saves a copy in Drive.
Studio retains every old model and lets you switch between generations.

This release trains a **bounded rational-quadratic spline flow**, with explicit
disease design-strength bands and coupled shape/material/disease controls.
Its control prior is shared across anatomy profiles because these authored samples
do not establish patient-population differences between diseases. The renderer
applies the selected organ, phenotype and independently seeded spatial pattern.
The separate datasets contain **13,824 baseline** and **69,120 disease**
authored examples. Paired surface observations include denser and lesion-focused queries.
Source templates, split families, recipes, phase seeds, licences and hashes are retained.
Full-source mesh checks are a separate export gate; surface-query records are not
independent patients or proof of collision-free anatomy for every possible recipe.

The open-evidence archive contains unpaired photographic references and measured 3D
case evidence. Eleven photographic ROI palettes contribute to rendered colour targets;
the new CT cases are reference material, not inputs to the training sampler or geometry
targets. Photographic RGB is not measured albedo; design severity is not clinical stage.
No tissue physics or new language model is trained by this notebook.


In [ ]:
"""Standard-library-only, checksum-bound expanded-disease Colab setup helpers."""

from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
import stat
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path, PurePosixPath


def file_digest(path: Path) -> str:
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def extract_package(archive_path: Path, expected: str, destination: Path, prefix: str) -> Path:
    if not re.fullmatch(r"[a-f0-9]{64}", expected) or file_digest(archive_path) != expected:
        raise ValueError("Archive does not match this notebook: " + archive_path.name)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination / prefix
    with zipfile.ZipFile(archive_path) as archive:
        entries = archive.infolist()
        names = [item.filename for item in entries]
        if (
            len(names) != len(set(names))
            or len(entries) > 10000
            or sum(i.file_size for i in entries) > 12_000_000_000
        ):
            raise ValueError("Archive entry count or expanded size is invalid")
        for item in entries:
            parts = item.filename.split("/")
            if (
                len(parts) < 2
                or parts[0] != prefix
                or any(p in ("", ".", "..") for p in parts)
                or "\\" in item.filename
                or ":" in item.filename
                or item.flag_bits & 1
                or item.is_dir()
                or stat.S_IFMT(item.external_attr >> 16) not in (0, stat.S_IFREG)
            ):
                raise ValueError("Unsafe ZIP member")
        manifest_name = prefix + "/package-manifest.json"
        manifest_bytes = archive.read(manifest_name)
        manifest = json.loads(manifest_bytes)
        hashes = manifest.get("artifact_hashes", {})
        if manifest.get("format") != "viscera-joint-package-v1" or not hashes:
            raise ValueError("Missing package manifest")
        if set(names) != {manifest_name} | {prefix + "/" + key for key in hashes}:
            raise ValueError("Package has extra or missing members")

        def verify(target: Path) -> None:
            if target.is_symlink() or any(p.is_symlink() for p in target.rglob("*")):
                raise ValueError("Extraction must not contain symlinks")
            if (target / "package-manifest.json").read_bytes() != manifest_bytes:
                raise ValueError("Cached package manifest differs from the pinned archive")
            actual = {p.relative_to(target).as_posix() for p in target.rglob("*") if p.is_file()}
            if actual != set(hashes) | {"package-manifest.json"}:
                raise ValueError("Extraction has extra or missing files")
            for name, sha in hashes.items():
                path = target / name
                if (
                    path.is_symlink()
                    or not path.resolve().is_relative_to(target.resolve())
                    or file_digest(path) != sha
                ):
                    raise ValueError("Extracted file checksum mismatch: " + name)

        if root.exists():
            verify(root)
            return root
        temporary = Path(tempfile.mkdtemp(prefix=".joint-extract-", dir=destination))
        try:
            for item in entries:
                target = temporary.joinpath(*PurePosixPath(item.filename).parts)
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(item) as source, target.open("xb") as output:
                    shutil.copyfileobj(source, output, 1024 * 1024)
            verify(temporary / prefix)
            (temporary / prefix).replace(root)
        finally:
            shutil.rmtree(temporary)
    return root


def install_environment(root: Path, environment: Path) -> Path:
    override = os.environ.get("VISCERA_TRAIN_PYTHON")
    # Resolving a venv's interpreter symlink would silently select the base
    # interpreter and install into the wrong environment (PEP 668 on macOS).
    python = Path(override).expanduser().absolute() if override else environment / "bin/python"
    if not override:
        if not python.is_file():
            result = subprocess.run([sys.executable, "-m", "venv", str(environment)])
            if result.returncode:
                subprocess.run(
                    [sys.executable, "-m", "pip", "install", "virtualenv==20.35.3"], check=True
                )
                subprocess.run([sys.executable, "-m", "virtualenv", str(environment)], check=True)
        host = subprocess.run(
            [
                sys.executable,
                "-c",
                "import torch,json; print(json.dumps({'version':torch.__version__.split('+')[0],'cuda':torch.version.cuda,'available':torch.cuda.is_available()}))",
            ],
            capture_output=True,
            text=True,
        )
        info = (
            json.loads(host.stdout)
            if host.returncode == 0
            else {"version": "2.11.0", "cuda": None, "available": False}
        )
        if not re.fullmatch(r"2\.\d+\.\d+", info["version"]):
            raise ValueError("Unsupported host PyTorch version")
        variant = (
            "cu" + info["cuda"].replace(".", "") if info["available"] and info["cuda"] else "cpu"
        )
        command = [
            str(python),
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "torch==" + info["version"],
        ]
        if sys.platform == "linux":
            command += ["--index-url", "https://download.pytorch.org/whl/" + variant]
        subprocess.run(command, check=True)
    subprocess.run(
        [
            str(python),
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "-r",
            str(root / "notebooks/requirements-joint-v2-colab.txt"),
        ],
        check=True,
    )
    subprocess.run(
        [
            str(python),
            "-c",
            "import numpy,scipy,torch,trimesh,pydantic; print('Joint training dependencies ready; CUDA:', torch.cuda.is_available())",
        ],
        check=True,
    )
    return python


def verify_weights(path: Path, receipt: dict) -> str:
    actual = file_digest(path)
    if receipt.get("status") != "complete" or receipt.get("weights_sha256") != actual:
        raise ValueError("Weight export did not complete or its checksum changed")
    with zipfile.ZipFile(path) as archive:
        entries = archive.infolist()
        names = [entry.filename for entry in entries]
        prefix = "viscera-joint-v2-weights/"
        if len(entries) > 1000 or sum(entry.file_size for entry in entries) > 512 * 1024 * 1024:
            raise ValueError("Joint weight archive exceeds its size limit")
        for entry in entries:
            parts = entry.filename.split("/")
            if (
                len(parts) < 2
                or parts[0] != prefix[:-1]
                or any(p in ("", ".", "..") for p in parts)
                or "\\" in entry.filename
                or ":" in entry.filename
                or entry.flag_bits & 1
                or entry.is_dir()
                or stat.S_IFMT(entry.external_attr >> 16) not in (0, stat.S_IFREG)
            ):
                raise ValueError("Unsafe weight archive member")
        manifest = json.loads(archive.read(prefix + "manifest.json"))
        hashes = manifest.get("artifact_hashes", {})
        required = {
            "model.npz",
            "model.json",
            "training_report.json",
            "exact-renderer-audit.json",
            "data/baseline-release.json",
            "data/disease-release.json",
        }
        if (
            manifest.get("format") != "viscera-joint-flow-v2"
            or manifest.get("smoke_only") is not False
            or manifest.get("profile") != "full"
            or not required <= hashes.keys()
            or len(names) != len(set(names))
            or set(names) != {prefix + "manifest.json"} | {prefix + k for k in hashes}
        ):
            raise ValueError("Invalid joint weight archive")
        for relative, expected in hashes.items():
            if hashlib.sha256(archive.read(prefix + relative)).hexdigest() != expected:
                raise ValueError("Weight member checksum mismatch")
        if hashlib.sha256(archive.read(prefix + "manifest.json")).hexdigest() != receipt.get(
            "export_manifest_sha256"
        ):
            raise ValueError("Export manifest changed after completion")
        audit = json.loads(archive.read(prefix + "exact-renderer-audit.json"))
        if audit.get("passed") is not True:
            raise ValueError("Exact renderer checks did not pass")
    return actual


PACKAGES = {'baseline': {'filename': 'viscera-joint-v2-baseline-2026-10-07-62ede316.zip', 'prefix': 'baseline', 'sha256': '3cd25ee646a74f961ed47aecbc4e8bea7583f9f6ff7e3668ee2829efb53c951f', 'bytes': 517320008, 'files': 10}, 'disease': {'filename': 'viscera-joint-v2-disease-2026-10-07-fdee30be.zip', 'prefix': 'disease', 'sha256': '094795bf00924b0c62b03bf4a18a182d1ced368bfcac776dbc87bd02f1938cce', 'bytes': 2646836294, 'files': 10}, 'runtime': {'filename': 'viscera-joint-v2-runtime-2026-10-07.zip', 'prefix': 'viscera', 'sha256': '4320f2ea427b87d7e06ac13e962ad35fae72ce1819b7b4dde539ad6c262e1704', 'bytes': 31480345, 'files': 117}}
DRIVE_FOLDER_NAME = 'VISCERA Expanded Disease Training RC2 2026-10-07'
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
if os.environ.get("VISCERA_JOINT_ARCHIVES"):
    DRIVE_FOLDER = Path(os.environ["VISCERA_JOINT_ARCHIVES"]).resolve()
elif IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_FOLDER = Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME
else:
    DRIVE_FOLDER = Path.cwd()
WORK = Path(os.environ.get("VISCERA_JOINT_WORK", "/content/viscera-joint-v2" if IS_COLAB else str(Path.cwd() / "viscera-joint-v2-work"))).resolve()
WORK.mkdir(parents=True, exist_ok=True)
PATHS = {}
for lane, package in PACKAGES.items():
    source = DRIVE_FOLDER / package["filename"]
    if not source.is_file():
        raise FileNotFoundError(f"Missing {source}. Keep all three training ZIPs in the named Drive folder.")
    local = WORK / package["filename"]
    if not local.exists() or file_digest(local) != package["sha256"]:
        shutil.copyfile(source, local)
    PATHS[lane] = extract_package(local, package["sha256"], WORK / package["sha256"][:12], package["prefix"])
    print("Verified", lane, package["sha256"])
ROOT = PATHS["runtime"]


## Prepare the isolated training environment

Setup preserves the host's PyTorch/CUDA wheel variant and installs pinned scientific
dependencies into a separate environment. The notebook kernel needs no restart.
The printed device identifies the GPU actually allocated, rather than assuming A100.


In [ ]:
TRAIN_PYTHON = install_environment(ROOT, WORK / "train-env")
RUNTIME = json.loads(subprocess.check_output([str(TRAIN_PYTHON), "-c", "import torch,sys,json; print(json.dumps({'python':sys.version.split()[0],'torch':torch.__version__,'cuda':torch.cuda.is_available(),'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))"], text=True))
run_identity = hashlib.sha256(json.dumps({"packages": PACKAGES, "runtime": RUNTIME}, sort_keys=True).encode()).hexdigest()[:16]
OUT = Path(os.environ.get("VISCERA_JOINT_OUT", str(DRIVE_FOLDER / "runs" / run_identity)))
OUT.mkdir(parents=True, exist_ok=True)
print("Runtime:", RUNTIME)
print("Outputs:", OUT)
if IS_COLAB and not RUNTIME["cuda"]:
    print("No GPU was allocated. CPU is supported, but switch to a GPU runtime for faster full training.")


## Train, compare held-out distributions, and audit full rendered meshes

Training uses a shared baseline/disease control prior and bounded invertible spline
couplings. Validation selects a checkpoint against the identity/uniform baseline;
the untouched audit split must also improve before weights can be exported.
The report retains per-profile/severity results even where a baseline performs better.
The query network measures how well surface fields can be predicted. Its loss is
diagnostic by default: the tested field regularizer did not improve held-out density.

After training, the exact renderer checks three seeds for baseline conditions and
three seeds in each mild/moderate/severe design band for every disease condition.
The ZIP is exported only after all required checks pass. This is engineering
verification, not medical or mechanical validation.

Completed training artifacts are verified before reuse. The training log and model
report remain in the Drive run directory. Errors do not produce a success receipt.


In [ ]:
command = [str(TRAIN_PYTHON), "-u", str(ROOT / "scripts/joint_v2_train.py"),
           "--baseline", str(PATHS["baseline"]), "--disease", str(PATHS["disease"]),
           "--templates", str(ROOT / "templates"), "--out", str(OUT), "--device", "auto"]
environment = dict(os.environ, PYTHONPATH=str(ROOT / "py") + os.pathsep + str(ROOT), PYTHONDONTWRITEBYTECODE="1")
with (OUT / "training.log").open("a") as log:
    process = subprocess.Popen(command, cwd=ROOT, env=environment, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    returncode = process.wait()
if returncode:
    raise RuntimeError(f"Training/export failed with code {returncode}. See {OUT / 'training.log'}; no successful weights are claimed.")
COMPLETION = json.loads((OUT / "completion.json").read_text())
MODEL = json.loads((OUT / "model/model.json").read_text())
print("Selected method:", MODEL["selected_method"])
print("Selected training step:", MODEL.get("selected_step"), "of", MODEL.get("completed_steps"))
print("Validation density:", json.dumps(MODEL.get("validation", {}).get("density", {}).get("macro", {}), indent=2))
print("Audit density:", json.dumps(MODEL.get("audit", {}).get("density", {}).get("macro", {}), indent=2))
print("Promotion checks:", json.dumps(MODEL.get("validation_acceptance", {}), indent=2))
print("Full per-profile reports:", OUT / "model/training_report.json")
print("Full renderer audit:", OUT / "exact-renderer-audit.json")


## Download and play in Studio

Import **viscera-joint-v2-weights.zip** in **Trained models → Validate & activate**.
Choose an organ and a supported disease, set **Disease design range**, then use
**Joint flow variation** or **New joint variation**. Shape, colour and disease vary
together. The displayed mesh remains editable and supports textured GLB export.

Use **Preserved joint models** to reactivate an earlier VAE or flow. Importing this
new generation does not erase older weights or retrain the separate text adapter.


In [ ]:
WEIGHTS = OUT / "viscera-joint-v2-weights.zip"
weights_sha = verify_weights(WEIGHTS, COMPLETION)
print("Verified weights:", WEIGHTS)
print("SHA-256:", weights_sha)
if IS_COLAB:
    from google.colab import files
    files.download(str(WEIGHTS))
else:
    print("Local full run complete. Import the displayed ZIP in Studio.")
